# ✅ GIADA Task 3c — Conferma `m+h` con training multi-seed vettorializzato
La configurazione simmetrica è congelata. Il fresh viene aperto una sola volta dopo il gate development.

In [ ]:
from pathlib import Path
import base64, json, shutil, subprocess, sys
from IPython.display import Javascript, display
WORK=Path('/kaggle/working/giada_task_3c'); GIADA_REPO=WORK/'giada'; TEACHER_REPO=WORK/'neuron_as_deep_net'
subprocess.run(['git','clone','https://github.com/Zagred47/giada.git',str(GIADA_REPO)],check=True)
subprocess.run(['git','-C',str(GIADA_REPO),'fetch','origin','codex/surrogate-validity-audit'],check=True)
subprocess.run(['git','-C',str(GIADA_REPO),'checkout','--detach','FETCH_HEAD'],check=True)
subprocess.run(['git','clone','https://github.com/SelfishGene/neuron_as_deep_net.git',str(TEACHER_REPO)],check=True)
subprocess.run(['git','-C',str(TEACHER_REPO),'checkout','--detach','074c4666300a8ad246601dab179a97a6942f0f29'],check=True)
REVISION=subprocess.check_output(['git','-C',str(GIADA_REPO),'rev-parse','HEAD'],text=True).strip(); print({'revision':REVISION})


In [ ]:
sys.path.insert(0,str(GIADA_REPO))
for name in [n for n in list(sys.modules) if n=='src' or n.startswith('src.')]: del sys.modules[name]
import torch
assert torch.cuda.is_available(),'La Task 3c preregistrata richiede una GPU CUDA Kaggle.'
from src.giada_teacher import ExtractedGateFormula,JointGateSymmetricConfirmationConfig,prepare_joint_gate_symmetric_confirmation,train_vectorized_symmetric_joint_gate,evaluate_frozen_symmetric_joint_gate
prereg=json.loads((GIADA_REPO/'experiments/teacher_joint_gate_symmetric_confirmation_preregistration_v1.json').read_text())
task3b=json.loads((GIADA_REPO/'experiments/teacher_joint_gate_optimization_diagnosis_result_v1.json').read_text())
assert task3b['interpretation']['primary_repair']=='symmetric m/h rate supervision'
display({'gpu':torch.cuda.get_device_name(0),'preregistration':prereg})


In [ ]:
OUTPUT_DIR=Path('/kaggle/working/artifacts/giada_joint_m_h_symmetric_confirmation')
assert not OUTPUT_DIR.exists(),f'Output già presente: {OUTPUT_DIR}. Avvia una sessione nuova.'
formula=ExtractedGateFormula.from_mod(TEACHER_REPO/'L5PC_NEURON_simulation/mods/Ca_HVA.mod')
bundle=prepare_joint_gate_symmetric_confirmation(formula); config=JointGateSymmetricConfirmationConfig()
display({'contract':bundle['confirmation_contract'],'seeds':config.seeds,'fixed_checkpoint':config.checkpoints[-1],'compile_requested':config.enable_torch_compile})


## 🚀 Training e freeze development
La cella esegue il preflight di equivalenza, poi addestra contemporaneamente i tre seed. Stampa soltanto ai checkpoint.

In [ ]:
training=train_vectorized_symmetric_joint_gate(bundle,OUTPUT_DIR,config,code_revision=REVISION)
last=training['checkpoints'][-1]
display({'valid':training['valid'],'equivalence':training['equivalence_preflight'],'compile':training['compile'],'elapsed_minutes':training['execution']['elapsed_seconds']/60,'model_steps_per_second':training['execution']['model_steps_per_second'],'development_mean':last['mean_score'],'development_max_seed':last['max_seed_score'],'fresh_accessed':training['fresh_accessed']})
assert training['valid'] and training['equivalence_preflight']['valid'] and not training['fresh_accessed']


## 🔒 Apertura fresh una sola volta
Eseguire soltanto dopo il completamento e il freeze della cella precedente.

In [ ]:
final=evaluate_frozen_symmetric_joint_gate(bundle,OUTPUT_DIR,config)
display({'valid':final['valid'],'decision':final['decision'],'selection_used_fresh':final['selection_used_fresh']})
assert final['valid'] and not final['selection_used_fresh']


## 📦 Download
Download Blob/base64 compatibile con Kaggle.

In [ ]:
archive=Path(shutil.make_archive('/kaggle/working/giada_joint_m_h_symmetric_confirmation','zip',OUTPUT_DIR.parent,OUTPUT_DIR.name))
payload=base64.b64encode(archive.read_bytes()).decode('ascii')
display(Javascript(f"""const b=atob('{payload}');const a=new Uint8Array(b.length);for(let i=0;i<b.length;i++)a[i]=b.charCodeAt(i);const u=URL.createObjectURL(new Blob([a],{{type:'application/zip'}}));const l=document.createElement('a');l.href=u;l.download='{archive.name}';document.body.appendChild(l);l.click();l.remove();setTimeout(()=>URL.revokeObjectURL(u),1000);"""))
print({'archive':archive.name,'size_mib':round(archive.stat().st_size/2**20,2)})
